# VoxCPM Text-to-Speech Setup
Install the required dependencies.

In [ ]:
!pip install voxcpm soundfile gradio modelscope

## Load Model
This may take some time depending on your internet connection.

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
import gradio as gr

print("Loading VoxCPM model... (This will download model weights if not already present)")
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print("Model loaded successfully!")

## Gradio User Interface
Run this cell to launch the UI. It will provide a local link and a public share link.

In [ ]:
def generate_speech(text, mode, control_prompt, ref_audio, prompt_audio, prompt_text):
    if not text.strip():
        return "Error: Please enter text to synthesize.", None
        
    output_path = "output.wav"
    
    try:
        if mode == "Text-to-Speech (No Reference)":
            wav = model.generate(
                text=text,
                cfg_value=2.0,
                inference_timesteps=10,
                seed=42,
            )
        elif mode == "Voice Design (Description)":
            if not control_prompt:
                return "Error: Please provide a voice description.", None
            full_text = f"({control_prompt}){text}"
            wav = model.generate(
                text=full_text,
                cfg_value=2.0,
                inference_timesteps=10,
                seed=42,
            )
        elif mode == "Voice Cloning (Reference Audio)":
            if not ref_audio:
                return "Error: Please upload a reference audio file.", None
            wav = model.generate(
                text=text,
                reference_wav_path=ref_audio,
            )
        elif mode == "Controllable Voice Cloning":
             if not ref_audio:
                return "Error: Please upload a reference audio file.", None
             full_text = f"({control_prompt}){text}" if control_prompt else text
             wav = model.generate(
                text=full_text,
                reference_wav_path=ref_audio,
                cfg_value=2.0,
                inference_timesteps=10,
                seed=42,
            )
        elif mode == "Ultimate Cloning":
            if not prompt_audio or not prompt_text:
                 return "Error: Please provide both reference transcript and prompt audio.", None
            wav = model.generate(
                text=text,
                prompt_wav_path=prompt_audio,
                prompt_text=prompt_text,
                reference_wav_path=ref_audio if ref_audio else prompt_audio,
            )
        else:
            return "Invalid mode selected.", None

        sf.write(output_path, wav, model.tts_model.sample_rate)
        return "Success!", output_path
    except Exception as e:
        return f"An error occurred: {str(e)}", None

with gr.Blocks(title="VoxCPM Text-to-Speech") as demo:
    gr.Markdown("# VoxCPM Text-to-Speech Generator")
    gr.Markdown("Based on [OpenBMB/VoxCPM](https://github.com/OpenBMB/VoxCPM). Select a generation mode and fill in the required inputs.")
    
    with gr.Row():
        with gr.Column():
            mode = gr.Dropdown(
                choices=[
                    "Text-to-Speech (No Reference)", 
                    "Voice Design (Description)",
                    "Voice Cloning (Reference Audio)",
                    "Controllable Voice Cloning",
                    "Ultimate Cloning"
                ],
                value="Text-to-Speech (No Reference)",
                label="Generation Mode"
            )
            text_input = gr.Textbox(lines=5, label="Text to Synthesize", placeholder="Enter the text you want to synthesize...")
            control_prompt = gr.Textbox(lines=2, label="Voice Description / Style Control (for Voice Design/Controllable Cloning)", placeholder="E.g. A young woman, gentle and sweet voice")
            
            with gr.Accordion("Audio Inputs (for Cloning)", open=False):
                ref_audio = gr.Audio(type="filepath", label="Reference Audio (for Voice Cloning / Ultimate Cloning)")
                prompt_audio = gr.Audio(type="filepath", label="Prompt Audio (for Ultimate Cloning)")
                prompt_text = gr.Textbox(lines=2, label="Reference Transcript (for Ultimate Cloning)", placeholder="Transcript of the prompt audio...")
            
            generate_btn = gr.Button("Generate Speech", variant="primary")
            
        with gr.Column():
            status_output = gr.Textbox(label="Status")
            audio_output = gr.Audio(label="Generated Audio")
            
    generate_btn.click(
        fn=generate_speech,
        inputs=[text_input, mode, control_prompt, ref_audio, prompt_audio, prompt_text],
        outputs=[status_output, audio_output]
    )

# Use share=True to create a public link, which is useful when running in Google Colab
demo.launch(share=True, debug=True)
